In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time

headers = {
    "User-Agent": "Mozilla/5.0"
}

# Starting pages
start_urls = [
    "https://en.wikipedia.org/wiki/Waste_management",
    "https://en.wikipedia.org/wiki/Recycling",
    "https://en.wikipedia.org/wiki/Plastic_pollution",
    "https://en.wikipedia.org/wiki/Electronic_waste",
    "https://en.wikipedia.org/wiki/Biodegradable_waste"
]

visited = set()
all_text = []

MAX_PAGES = 50   # increase for larger dataset

def scrape_page(url):
    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return [], []

        soup = BeautifulSoup(response.text, "html.parser")

        # Extract paragraphs
        texts = []

        for p in soup.find_all("p"):
            text = p.get_text(strip=True)

            if len(text) > 100:
                texts.append(text)

        # Extract internal wiki links
        links = []

        for a in soup.find_all("a", href=True):
            href = a["href"]

            if href.startswith("/wiki/") and ":" not in href:
                full_link = "https://en.wikipedia.org" + href
                links.append(full_link)

        return texts, links

    except Exception as e:
        print("Error:", e)
        return [], []


queue = start_urls.copy()

while queue and len(visited) < MAX_PAGES:

    url = queue.pop(0)

    if url in visited:
        continue

    print(f"Scraping: {url}")

    visited.add(url)

    texts, links = scrape_page(url)

    all_text.extend(texts)

    # Add new links
    for link in links:
        if link not in visited:
            queue.append(link)

    time.sleep(1)


# Create DataFrame
df = pd.DataFrame({
    "text": all_text
})

# Remove duplicates
df.drop_duplicates(inplace=True)

# Save
df.to_csv("big_waste_dataset.csv", index=False)

print("\nDataset saved!")
print("Total samples:", len(df))

Scraping: https://en.wikipedia.org/wiki/Waste_management
Scraping: https://en.wikipedia.org/wiki/Recycling
Scraping: https://en.wikipedia.org/wiki/Plastic_pollution
Scraping: https://en.wikipedia.org/wiki/Electronic_waste
Scraping: https://en.wikipedia.org/wiki/Biodegradable_waste
Scraping: https://en.wikipedia.org/wiki/Main_Page
Scraping: https://en.wikipedia.org/wiki/Waste_Management_(corporation)
Scraping: https://en.wikipedia.org/wiki/Waste_management_(disambiguation)
Scraping: https://en.wikipedia.org/wiki/Garbage_disposal_unit
Scraping: https://en.wikipedia.org/wiki/Sanitary_engineering
Scraping: https://en.wikipedia.org/wiki/Garbage_truck
Scraping: https://en.wikipedia.org/wiki/Waste_collector
Scraping: https://en.wikipedia.org/wiki/Stockholm
Scraping: https://en.wikipedia.org/wiki/Waste_picker
Scraping: https://en.wikipedia.org/wiki/Agbogbloshie
Scraping: https://en.wikipedia.org/wiki/Waste_container
Scraping: https://en.wikipedia.org/wiki/Gda%C5%84sk_University_of_Technology
S